In [1]:
import pandas as pd
import xgboost as xgb
import numpy as np
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.metrics import f1_score

In [2]:
# Load the dataset
df = pd.read_csv(r'C:\Users\khiew\Downloads\FYP Reduced Dataset.csv')

In [3]:
# Drop diseases with less than 1000 instances
disease_counts = df['diseases'].value_counts()
valid_diseases = disease_counts[disease_counts >= 1000].index
df = df[df['diseases'].isin(valid_diseases)]

# Assuming that the target variable is 'diseases' and all other variables are input features
X = df.drop('diseases', axis=1)
y = df['diseases']

# Encode the target variable (diseases) if it's a categorical variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
print("Number of remaining classes in training set:", len(np.unique(y_train)))
print("Number of rows left:", len(df))

Number of remaining classes in training set: 37
Number of rows left: 44748


In [4]:
# Instead, calculate class weights inversely proportional to class frequencies
class_counts = pd.Series(y_train).value_counts()
total_samples = len(y_train)
class_weights = {class_idx: total_samples / (len(class_counts) * count) 
                for class_idx, count in class_counts.items()}

In [6]:
def objective(trial):
    params = {
        'objective': 'multi:softprob',
        'num_class': len(np.unique(y_train)),
        'tree_method': 'hist',
        'eval_metric': 'mlogloss',
        
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
        
        # Try including early stopping directly in the model parameters
        'early_stopping_rounds': 50,
    }
    
    # Create XGBoost model with parameters from Optuna
    model = xgb.XGBClassifier(**params)
    
     # Create sample_weight array based on class weights
    sample_weight = np.array([class_weights[y] for y in y_train])
    
    model.fit(
        X_train,  # Use original training data instead of resampled
        y_train,  # Use original labels instead of resampled
        eval_set=[(X_test, y_test)],
        sample_weight=sample_weight,  # Add sample weights
        verbose=False
    )
    
    preds = model.predict(X_test)
    accuracy = accuracy_score(y_test, preds)
    return accuracy

In [8]:
# Create Optuna study for optimization with persistent storage
study = optuna.create_study(
    direction='maximize',  # Assuming you are maximizing accuracy
    study_name="XGboost_diseases_symptoms_dropextremelymore1000withoutSMOTE_study", 
    storage=r"sqlite:///C:/Users/khiew/Downloads/xgboost.db", 
    load_if_exists=True  # Load the study if it already exists, to resume from the last trial
)

# Optimize the study with your objective function, you can adjust the n_trials as needed
study.optimize(objective, n_trials=20)
# Print the best trial and hyperparameters
print("\nBest Trial:")
print(study.best_trial)
print("Best Hyperparameters:")
print(study.best_trial.params)

[I 2025-04-23 12:26:25,703] A new study created in RDB with name: XGboost_diseases_symptoms_dropextremelymore1000withoutSMOTE_study
[I 2025-04-23 12:27:05,246] Trial 0 finished with value: 0.6912849162011173 and parameters: {'max_depth': 14, 'learning_rate': 0.14735199873190788, 'n_estimators': 721, 'subsample': 0.8199516419324785, 'colsample_bytree': 0.8727500579770675, 'gamma': 4.481572822964834, 'reg_alpha': 3.2576516026396236, 'reg_lambda': 0.3427546998872394}. Best is trial 0 with value: 0.6912849162011173.
[I 2025-04-23 12:27:41,932] Trial 1 finished with value: 0.6896089385474861 and parameters: {'max_depth': 12, 'learning_rate': 0.0675627815433758, 'n_estimators': 383, 'subsample': 0.9937640075488491, 'colsample_bytree': 0.9929528691160628, 'gamma': 0.35940091209419656, 'reg_alpha': 3.8430343109046445, 'reg_lambda': 4.648781480435743}. Best is trial 0 with value: 0.6912849162011173.
[I 2025-04-23 12:28:36,295] Trial 2 finished with value: 0.6894972067039106 and parameters: {'ma


Best Trial:
FrozenTrial(number=9, state=TrialState.COMPLETE, values=[0.6959776536312849], datetime_start=datetime.datetime(2025, 4, 23, 12, 31, 14, 961598), datetime_complete=datetime.datetime(2025, 4, 23, 12, 31, 27, 861762), params={'max_depth': 12, 'learning_rate': 0.22012667241676706, 'n_estimators': 201, 'subsample': 0.9218739984398711, 'colsample_bytree': 0.6299642566377859, 'gamma': 3.319086639463023, 'reg_alpha': 1.1422964084598308, 'reg_lambda': 4.8892209318150615}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'max_depth': IntDistribution(high=15, log=False, low=3, step=1), 'learning_rate': FloatDistribution(high=0.3, log=False, low=0.01, step=None), 'n_estimators': IntDistribution(high=1000, log=False, low=100, step=1), 'subsample': FloatDistribution(high=1.0, log=False, low=0.5, step=None), 'colsample_bytree': FloatDistribution(high=1.0, log=False, low=0.5, step=None), 'gamma': FloatDistribution(high=5.0, log=False, low=0.0, step=None), 'reg_alpha'